# A04 — mnist: dữ liệu và so sánh đã lưu

Các framework dùng hai môi trường WSL riêng. Mã huấn luyện ở `training/mnist_keras.ipynb` và `training/mnist_torch.ipynb`, cùng `training/mnist_scratch.ipynb`. Notebook này đọc kết quả gốc; Run All không train/TEST. Toàn bộ notebook huấn luyện nguyên byte và output cũ nằm trong provenance.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'torch')
_migration


# 1. Assignment Overview


Notebook này xây pipeline Assignment 04 cho đúng bài toán CNN image classification trên MNIST: ảnh chữ số viết tay 28x28 grayscale, 10 classes. Notebook tập trung vào representation ảnh, convolution operations, NumPy CNN from scratch, Keras CNN, PyTorch CNN, so sánh framework và phân tích lỗi dự đoán.

**Nhận xét kết quả.**

- Bảng overview xác nhận notebook dùng MNIST image data thật, không còn dùng tabular dataset.
- Các artifact của phase MNIST được tách dưới `pipeline04/mnist/`.

# 2. Environment Setup


Import thư viện, cố định seed, kiểm tra version TensorFlow/PyTorch và xác định device. Output chỉ dùng thông tin môi trường cần thiết, không in path cá nhân.

**Nhận xét kết quả.**

- Output cho biết notebook đang chạy bằng CPU hay GPU và version các framework.
- Seed được cố định để các split, initialization và DataLoader reproducible hơn.

# 3. Load Dataset


Load MNIST từ file cache trong `datasets/mnist/`. Nếu file chưa có, code có cơ chế tải lại từ TensorFlow/Keras official archive.

**Nhận xét kết quả.**

- MNIST cache nằm tại `datasets/mnist/mnist.npz`, kích thước khoảng 10.96 MB.
- Dataset load được 60,000 ảnh train gốc và 10,000 ảnh test; mỗi ảnh là grayscale 28x28 với 10 class.
- Pixel ban đầu ở thang 0-255, nên cần normalize trước khi train CNN.

# 4. Dataset Inspection


Kiểm tra phân phối class, hiển thị sample images và inspect một ảnh cụ thể để hiểu dữ liệu ảnh trước khi chuẩn hóa.

**Nhận xét kết quả.**

- Class distribution giúp kiểm tra MNIST có đủ 10 chữ số và không bị lệch class nghiêm trọng.
- Các grid ảnh cho thấy input là chữ số viết tay grayscale, đúng loại dữ liệu mà Conv2D có cơ sở dùng.

# 5. Data Preparation


Tạo train/validation/test split, normalize pixel và chuẩn bị hai layout tensor: Keras dùng channels-last, PyTorch dùng channels-first.

**Nhận xét kết quả.**

- Validation được lấy từ original training set: train 54,000, validation 6,000, test 10,000.
- Normalize `/255` không học tham số từ validation/test, nên không tạo leakage.
- Keras dùng tensor `(B, 28, 28, 1)`, còn PyTorch dùng `(B, 1, 28, 28)`.

# 6. Discover CNN Operations


Chạy các operation CNN cơ bản trên tensor nhỏ: Conv2D, ReLU, MaxPool, Flatten, Dense, logits và softmax.

**Nhận xét kết quả.**

- Conv2D giữ batch dimension và đổi channel từ 1 sang số filter.
- MaxPool giảm height/width, Flatten chuyển feature map thành vector, Dense tạo 10 logits cho 10 class.
- Softmax sum xấp xỉ 1, đúng tính chất phân phối xác suất.

# 7. Inspect CNN Building Blocks


Dùng một batch MNIST thật để kiểm tra shape qua từng building block trong cả Keras và PyTorch.

**Nhận xét kết quả.**

- Một batch có 8 ảnh tạo ra 8 hàng logits, mỗi hàng có 10 giá trị cho 10 chữ số.
- Keras dùng `(B, H, W, C)`, còn PyTorch dùng `(B, C, H, W)`.

# 8. CNN From Scratch with NumPy


Định nghĩa các layer NumPy có forward/backward thật: Conv2D, ReLU, MaxPool2D, Flatten, Dense, SoftmaxCrossEntropy và SGD update.

**Nhận xét kết quả.**

- Các class NumPy đã có `forward()` và `backward()` thật cho Conv2D, ReLU, MaxPool2D, Flatten, Dense và SoftmaxCrossEntropy.
- `SGDNumPy` thực hiện bước update tham số bằng gradient đã tính thủ công, không dùng TensorFlow/PyTorch autograd.
- Conv2D dùng im2col/col2im để giữ code dễ kiểm tra hơn so với nhiều vòng lặp pixel lồng nhau.

# 9. NumPy CNN Training


Test từng NumPy layer, tạo CNN nhỏ, train thật bằng forward/loss/backward/update trên stratified subset và lưu best state.

**Nhận xét kết quả.**

- Layer tests kiểm tra output shape, gradient shape, finite values, parameter shape và softmax row sums trước khi train.
- Numerical gradient check trên Conv2D đối chiếu gradient backward với sai phân hữu hạn cho một weight nhỏ.
- Tiny-batch test xác nhận loss giảm sau vài bước SGD, rồi NumPy CNN train thật trên 6,000 samples và lưu best state theo validation Macro-F1.

# 10. Tensor Shape and Parameter Inspection


Tính parameter từng layer của NumPy CNN và đối chiếu shape flow qua mạng.

**Nhận xét kết quả.**

- Conv1 có `(3 x 3 x 1 x 2) + 2` parameters: 18 weights và 2 bias.
- Tổng parameters nhỏ để NumPy training khả thi, nhưng vẫn có Conv, ReLU, Pool, Flatten và Dense đúng pipeline CNN.

# 11. CNN with Keras/TensorFlow


Xây Keras Basic CNN tương đương về ý tưởng với NumPy CNN nhưng dùng full framework để train hiệu quả hơn.

**Nhận xét kết quả.**

- `model.summary()` cho thấy spatial size giảm sau mỗi MaxPool: 28x28 → 14x14 → 7x7.
- Channel tăng từ 1 lên 8 rồi 16 nhờ Conv2D filters.
- Output cuối là 10 probabilities cho 10 digit classes.

# 12. Keras Training


Train Keras Basic CNN với validation, checkpoint, EarlyStopping và lưu training curves.

**Nhận xét kết quả.**

- Keras evaluation dùng test split độc lập của MNIST và đạt Accuracy 0.9820, Macro-F1 0.9819.
- Macro metrics đánh giá trung bình giữa 10 chữ số, còn confusion matrix cho biết digit nào bị nhầm với digit nào.

# 13. Keras Evaluation


Load Keras checkpoint tốt nhất, evaluate trên test set và lưu metrics/confusion matrix.

**Nhận xét kết quả.**

- Keras evaluation dùng test split độc lập của MNIST.
- Macro metrics đánh giá trung bình giữa 10 chữ số, còn confusion matrix cho biết digit nào bị nhầm với digit nào.

# 14. CNN with PyTorch


Xây PyTorch CNN tương đương về ý tưởng với Keras Basic CNN và kiểm tra input/output shape trước khi train.

**Nhận xét kết quả.**

- PyTorch nhận ảnh theo layout `(B, C, H, W)`.
- Output logits có shape `(B, 10)`, nghĩa là mỗi ảnh có 10 score chưa softmax.

# 15. PyTorch Training


Train PyTorch CNN bằng loop thủ công có zero grad, forward, loss, backward và optimizer step.

**Nhận xét kết quả.**

- Training loop hiển thị rõ các bước gradient thủ công của PyTorch.
- Best checkpoint được chọn theo validation Macro-F1 để cân bằng chất lượng giữa các class.

# 16. PyTorch Evaluation


Load PyTorch checkpoint tốt nhất và evaluate trên test set bằng cùng protocol metrics với Keras.

**Nhận xét kết quả.**

- Bảng so sánh cùng các cột Accuracy, Macro-P, Macro-R, Macro-F1, Params, Train Samples, Best Epoch, Train Time và Prediction Time.
- NumPy CNN đạt Macro-F1 0.7733 với subset nhỏ; Keras đạt 0.9819 và PyTorch đạt 0.9766 trên full training split.
- Không nên kết luận framework nào luôn tốt hơn từ một run; runtime phụ thuộc hardware/backend và mức tối ưu của implementation.

# 17. Framework Comparison


Evaluate NumPy CNN trên test set và so sánh NumPy, Keras, PyTorch bằng cùng metric classification.

**Nhận xét kết quả.**

- Bảng so sánh cùng các cột Accuracy, Macro-P, Macro-R, Macro-F1, Params, Train Samples, Best Epoch, Train Time và Prediction Time.
- NumPy CNN đạt Macro-F1 0.7733 với subset nhỏ; Keras đạt 0.9819 và PyTorch đạt 0.9766 trên full training split.
- Không nên kết luận framework nào luôn tốt hơn từ một run; runtime phụ thuộc hardware/backend và mức tối ưu của implementation.

# 18. CNN Architecture Improvements


Thử cải tiến nhẹ trên MNIST: M1 Basic CNN và M2 Basic CNN + BatchNorm + Dropout.

**Nhận xét kết quả.**

- Bảng test metrics cho biết kết quả cuối của từng model sau khi đã train.
- `selection_results` dùng validation Macro-F1 để chọn final model, nên test set không bị dùng để tune/chọn model.
- MNIST dễ nên các framework CNN đạt kết quả cao; NumPy CNN nhỏ có mục tiêu giáo dục hơn là tối ưu performance.

# 19. Improved Model Comparison


Tổng hợp Basic CNN, improved CNN và chọn candidate tốt nhất theo Macro-F1.

**Nhận xét kết quả.**

- Bảng này chọn model tốt nhất theo Macro-F1 sau khi có NumPy, Keras, PyTorch và improved Keras.
- Model tốt nhất trong run này là `M2 Basic CNN + BatchNorm + Dropout` với Accuracy 0.9851 và Macro-F1 0.9850.
- MNIST dễ nên các framework CNN đạt kết quả cao; NumPy CNN nhỏ có mục tiêu giáo dục hơn là tối ưu performance.

# 20. Feature Map Inspection


Lấy ảnh MNIST thật và hiển thị feature maps từ block Conv đầu và block Conv thứ hai của Keras model tốt nhất.

**Nhận xét kết quả.**

- Feature maps là activation thật từ Conv2D layers trên ảnh MNIST.
- Notebook không gán cứng một channel là edge detector; output chỉ cho thấy mỗi filter phản ứng khác nhau với nét chữ.

# 21. Error Analysis


Hiển thị các dự đoán sai có confidence cao nhất và dùng confusion matrix để xem digit nào dễ nhầm.

**Nhận xét kết quả.**

- Error examples hiển thị ảnh thật, true label, predicted label và confidence.
- Confusion pair lớn nhất trong run này là true `4` bị dự đoán thành `9` với 13 cases.
- Các lỗi này thường liên quan đến nét viết mơ hồ hoặc chữ số có hình dáng gần nhau.

# 22. Final Model


Chọn model cuối theo Macro-F1, lưu final metrics, final model reference và confusion matrix cuối.

**Nhận xét kết quả.**

- Final model được chọn bằng validation Macro-F1, sau đó mới đọc test metrics tương ứng để báo cáo.
- Model cuối được lưu riêng để dễ tìm khi viết báo cáo hoặc demo.
- Metrics cuối là kết quả thật từ model đã chọn, không phải số điền trước.

# 23. Final Results


Lưu benchmark, metadata và liệt kê toàn bộ artifacts MNIST CNN đã tạo.

**Nhận xét kết quả.**

- Bảng benchmark cuối tổng hợp kết quả thật của NumPy CNN, Keras CNN, PyTorch CNN và improved CNN.
- Artifact listing xác nhận notebook đã lưu 5 model files, 20 metric files và 13 figures dưới `pipeline04/mnist/`.
- Tổng recorded train time hiện tại khoảng 58.33 giây.

In [ ]:
import pandas as pd
from IPython.display import display, Image
_folder = _repo / 'A04/artifacts/pipeline04/mnist/metrics'
for _file in sorted(_folder.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))


In [ ]:
display(Image(filename=str(_repo / 'A04/artifacts/pipeline04/mnist/figures/keras_basic_training_curves.png')))


In [ ]:
display(Image(filename=str(_repo / 'A04/artifacts/pipeline04/mnist/figures/m2_batchnorm_dropout_training_curves.png')))


In [ ]:
display(Image(filename=str(_repo / 'A04/artifacts/pipeline04/mnist/figures/pytorch_basic_training_curves.png')))
